### Agent V: Custom Tool Agent - Practice & Exercises
In this notebook, you will practice building an agent with custom tools and loop-backs to execute complex actions like document drafting and todo list management.

#### Part 1: Follow-Along Practice
##### GoalBuild a document drafting agent (Drafter) that loops using custom tools (`update` and `save`) to modify and save content to a local file.

##### Steps:1. Define the `AgentState` schema using `add_messages` to append messages to history.
2. Create custom tools `@tool` for `update` (updates content in global variable) and `save` (saves content to a file).
3. Write the `our_agent` node function and `should_continue` conditional router function.
4. Compile and run the agent.

In [ ]:
from typing import Annotated, Sequence, TypedDict
from dotenv import load_dotenv
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode

load_dotenv()

document_content = ""

# 1. Define State
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

In [ ]:
# 2. Create custom tools
@tool
def update(content: str) -> str:
    """Updates the document with the provided content."""
    global document_content
    document_content = content
    return f"Document updated successfully. Current content is:\n{document_content}"

@tool
def save(filename: str) -> str:
    """Save the current document to a text file and finish.
    
    Args:
        filename: Name for the text file.
    """
    global document_content
    if not filename.endswith('.txt'):
        filename = f"{filename}.txt"
    try:
        with open(filename, 'w') as file:
            file.write(document_content)
        return f"Document has been saved successfully to '{filename}'."
    except Exception as e:
        return f"Error saving document: {str(e)}"

tools = [update, save]
model = ChatOpenAI(model="gpt-4o").bind_tools(tools)

In [ ]:
# 3. Define Node and Router
def our_agent(state: AgentState) -> dict:
    sys = SystemMessage(content=f"""
    You are Drafter, a helpful writing assistant. Use 'update' to modify content, and 'save' to finish.
    The current document content is: {document_content}
    """)
    response = model.invoke([sys] + list(state["messages"]))
    return {"messages": [response]}

def should_continue(state: AgentState) -> str:
    messages = state["messages"]
    for msg in reversed(messages):
        if isinstance(msg, ToolMessage) and "saved successfully" in msg.content:
            return "end"
    return "continue"

In [ ]:
# 4. Build, Compile & Run
graph = StateGraph(AgentState)
graph.add_node("agent", our_agent)
graph.add_node("tools", ToolNode(tools))
graph.set_entry_point("agent")
graph.add_edge("agent", "tools")
graph.add_conditional_edges(
    "tools",
    should_continue,
    {
        "continue": "agent",
        "end": END
    }
)
app = graph.compile()

---

#### Part 2: Practice Exercises
##### Exercise 1: Todo List Manager Agent
###### GoalBuild a Todo List Manager Agent with custom tools to add items, mark them as complete, and save the todo list to a text file.

###### Sample InputSequential inputs:
1. `"Add buy milk to my todo list."`
2. `"Save my todo list to todos.txt."`

###### Sample Output1. `"Todo list updated successfully. Current todos: ['buy milk']"`
2. `"Todo list has been saved successfully to 'todos.txt'."`

###### Plan1. **Define State & Global List:** Create an `AgentState` schema for messages and a global list `todo_list = []` to store items.
2. **Define Custom Tools:** Create `@tool` for `add_todo(todo: str)` (appends to list) and `save_todos(filename: str)` (writes list to file).
3. **Build Graph:** Initialize `StateGraph`, register agent and tool nodes, add loops, compile, and run.

In [ ]:
# TODO: Define your State, Tools, Nodes, Graph, Compile, and Invoke it here


<details>
<summary>💡 Click to see solution</summary>

```python
from typing import Annotated, Sequence, TypedDict
from langchain_core.messages import BaseMessage, SystemMessage, ToolMessage
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode

todo_list = []

### 1. Stateclass AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

### 2. Tools@tool
def add_todo(todo: str) -> str:
    """Adds a new todo item to the list."""
    global todo_list
    todo_list.append(todo)
    return f"Todo list updated successfully. Current todos: {todo_list}"

@tool
def save_todos(filename: str) -> str:
    """Saves the todo list to a file and finishes."""
    global todo_list
    if not filename.endswith('.txt'):
        filename = f"{filename}.txt"
    try:
        with open(filename, 'w') as f:
            f.write('\n'.join(todo_list))
        return f"Todo list has been saved successfully to '{filename}'."
    except Exception as e:
        return f"Error saving todos: {str(e)}"

tools = [add_todo, save_todos]
model = ChatOpenAI(model="gpt-4o").bind_tools(tools)

### 3. Nodes and Routingdef todo_agent(state: AgentState) -> dict:
    sys = SystemMessage(content=f"""
    You are a Todo List Manager. Use 'add_todo' to add items, and 'save_todos' to save and finish.
    Current todos: {todo_list}
    """)
    response = model.invoke([sys] + list(state["messages"]))
    return {"messages": [response]}

def check_continue(state: AgentState) -> str:
    messages = state["messages"]
    for msg in reversed(messages):
        if isinstance(msg, ToolMessage) and "saved successfully" in msg.content:
            return "end"
    return "continue"

### 4. Build Graphgraph = StateGraph(AgentState)
graph.add_node("agent", todo_agent)
graph.add_node("tools", ToolNode(tools))
graph.set_entry_point("agent")
graph.add_edge("agent", "tools")
graph.add_conditional_edges(
    "tools",
    check_continue,
    {
        "continue": "agent",
        "end": END
    }
)
app = graph.compile()

### Teststate = {"messages": [HumanMessage(content="Add buy milk to my todo list.")]}
state = app.invoke(state)
state["messages"].append(HumanMessage(content="Save my todo list to todos.txt."))
res = app.invoke(state)
print(res["messages"][-1].content)
```
</details>